In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =============================================
# Comprehensive EDA Notebook for Dataset
# =============================================

# Importing standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
from pandas.api.types import CategoricalDtype

# Optional: for interactive plots
import plotly.express as px

In [ ]:
# ---------------------------------------------
# Load dataset
# ---------------------------------------------
data_path = '/kaggle/input/e-commerce-customer-behaviour-dataset/E-commerce.csv'
df = pd.read_csv(data_path)

# Quick overview
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())


In [ ]:
# ---------------------------------------------
# Basic statistics and info
# ---------------------------------------------
print("\nData Info:")
df.info()

print("\nStatistical Summary (Numerical Columns):")
display(df.describe(include='all'))

print("\nStatistical Summary (Categorical Columns):")
display(df.describe(include='object'))


In [ ]:
# ---------------------------------------------
# Missing values
# ---------------------------------------------
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing_values / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing_values, 'missing_percent': missing_percent})
display(missing_df)



In [ ]:
# ---------------------------------------------
# Data types and categorical analysis
# ---------------------------------------------
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

# Value counts for categorical variables
for col in categorical_cols:
    print(f"\nColumn: {col}")
    display(df[col].value_counts().head(10))  # top 10 categories

In [ ]:
# -----------------------------
# Visualizations
# -----------------------------
numeric_features = ['Age', 'Annual Income', 'Time on Site']
categorical_features = ['Gender', 'Location']

for col in numeric_features:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f'{col} Distribution')
    plt.show()

for col in categorical_features:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=df)
    plt.title(f'{col} Counts')
    plt.show()

In [ ]:
# ---------------------------------------------
# Univariate Analysis
# ---------------------------------------------
# Numerical distributions
for col in numerical_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[col], kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {col}')
    plt.show()

# Categorical distributions
for col in categorical_cols:
    plt.figure(figsize=(10,4))
    sns.countplot(y=col, data=df, order=df[col].value_counts().index, palette='Set2')
    plt.title(f'Count Plot of {col}')
    plt.show()

In [ ]:
# ---------------------------------------------
# Correlation Analysis (Numerical Features)
# ---------------------------------------------
plt.figure(figsize=(12,8))
corr = df[numerical_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ---------------------------------------------
# Pairplots and joint plots (optional for small datasets)
# ---------------------------------------------
if len(numerical_cols) <= 6:  # avoid very large pairplots
    sns.pairplot(df[numerical_cols], diag_kind='kde', corner=True)
    plt.show()

In [ ]:
# ---------------------------------------------
# Domain-Specific Analysis / Modal Analysis
# ---------------------------------------------
# Example: If you have 'category', 'sub_category', 'price', 'quantity'
# This will change based on your dataset domain
if 'category' in df.columns and 'price' in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x='category', y='price', data=df)
    plt.title("Price Distribution Across Categories")
    plt.show()


# Modal analysis: most frequent values per categorical column
modal_df = pd.DataFrame(columns=['Column', 'Mode', 'Mode_Count', 'Mode_Percent'])
for col in df.select_dtypes(include=['object', 'category', 'int', 'float']).columns:
    mode_val = df[col].mode()[0]
    count = df[col].value_counts()[mode_val]
    percent = (count / len(df) * 100).round(2)
    modal_df.loc[len(modal_df)] = [col, mode_val, count, percent]
display(modal_df)


In [ ]:
# ---------------------------------------------
# Data Modeling Preparation
# ---------------------------------------------
# Basic preprocessing steps
# 1. Handling missing values
# 2. Encoding categorical variables
# 3. Scaling numerical variables


from sklearn.preprocessing import LabelEncoder, StandardScaler
# Convert categorical features
le_gender = LabelEncoder()
df['Gender_encoded'] = le_gender.fit_transform(df['Gender'])

le_location = LabelEncoder()
df['Location_encoded'] = le_location.fit_transform(df['Location'])

In [ ]:
# Purchase history features
def extract_purchase_features(purchase_list):
    """Calculate total spend and total purchases."""
    try:
        purchases = json.loads(purchase_list.replace("'", '"'))
        total_spent = sum([item['Price'] for item in purchases])
        total_items = len(purchases)
    except:
        total_spent = 0
        total_items = 0
    return pd.Series([total_spent, total_items])

df[['Total_Spent', 'Total_Purchases']] = df['Purchase History'].apply(extract_purchase_features)


# Browsing history features
def extract_browsing_features(browsing_list):
    try:
        views = json.loads(browsing_list.replace("'", '"'))
        total_views = len(views)
    except:
        total_views = 0
    return total_views

df['Total_Views'] = df['Browsing History'].apply(extract_browsing_features)

# Product review rating
def extract_review_rating(review_list):
    try:
        review = json.loads(review_list.replace("'", '"'))
        if isinstance(review, dict) and 'Rating' in review:
            return review['Rating']
        elif isinstance(review, dict):
            return list(review.values())[0]['Rating']
    except:
        return np.nan
df['Review_Rating'] = df['Product Reviews'].apply(extract_review_rating).fillna(0)

# Derived target for purchase likelihood
df['Made_Purchase'] = (df['Total_Purchases'] > 0).astype(int)

In [ ]:
# =============================
# Import Libraries
# =============================

import json


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder


In [ ]:
# =============================
# Predictive Models
# =============================

# Split features and target
features_classification = ['Age', 'Gender_encoded', 'Location_encoded', 'Annual Income', 'Time on Site', 'Total_Views', 'Review_Rating']
X = df[features_classification]
y_purchase = df['Made_Purchase']
y_clv = df['Total_Spent']
y_rating = df['Review_Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y_purchase, test_size=0.2, random_state=42)

In [ ]:
# -----------------------------
# 6a. Purchase Likelihood
# -----------------------------
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Purchase Likelihood Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
# -----------------------------
# 6b. Customer Lifetime Value (CLV) Prediction
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_clv, test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_train, y_train)
y_pred_clv = reg.predict(X_test)
print("CLV RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_clv)))
print("CLV R2 Score:", r2_score(y_test, y_pred_clv))

In [ ]:
# -----------------------------
# 6c. Product Rating Prediction
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_rating, test_size=0.2, random_state=42)
reg_rating = RandomForestRegressor(n_estimators=100, random_state=42)
reg_rating.fit(X_train, y_train)
y_pred_rating = reg_rating.predict(X_test)
print("Rating RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rating)))
print("Rating R2 Score:", r2_score(y_test, y_pred_rating))

In [ ]:

# -----------------------------
# 6d. Customer Segmentation
# -----------------------------
X_seg = df[['Age', 'Annual Income', 'Time on Site', 'Total_Spent', 'Total_Purchases', 'Total_Views']]
kmeans = KMeans(n_clusters=3, random_state=42)
df['Segment'] = kmeans.fit_predict(X_seg)
sns.countplot(x='Segment', data=df)
plt.title('Customer Segments')
plt.show()

from sklearn.decomposition import PCA

X_seg = df[['Age', 'Annual Income', 'Time on Site', 'Total_Spent', 'Total_Purchases', 'Total_Views']]
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_seg)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df['Segment'], palette='Set2', s=100)
plt.title("Customer Segments Visualization (PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()


In [ ]:
# -----------------------------
# 6e. Predict Churn / Repeat Purchase
# -----------------------------
# Define churn: customers with Total_Purchases == 0
df['Churn'] = (df['Total_Purchases'] == 0).astype(int)
X = df[features_classification]
y_churn = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y_churn, test_size=0.2, random_state=42)
clf_churn = RandomForestClassifier(n_estimators=100, random_state=42)
clf_churn.fit(X_train, y_train)
y_pred_churn = clf_churn.predict(X_test)
print("Churn Prediction Accuracy:", accuracy_score(y_test, y_pred_churn))
print(classification_report(y_test, y_pred_churn))